# Week 3: noise, smoothing and edges

**What we do.** We add Gaussian noise to a photograph, remove it with the Gaussian (averaging) filter and with the median filter, measure the quality with the PSNR, and detect edges with the Canny algorithm.

**What to expect.** Both filters reduce noise, but the Gaussian filter also blurs edges, whereas the median filter keeps them sharp. Canny produces thin, continuous edge lines.

**How to read the code.** Functions come from three modules of `wavelets_lab`: `images` (loading, display), `filters` (noise, smoothing, PSNR) and `edges` (gradient, Canny). The last section of the notebook describes all of them.

In [ ]:
import matplotlib.pyplot as plt

from wavelets_lab import edges, filters, images

## Gaussian noise

**What we do.** We add white Gaussian noise with standard deviation 20 to the `camera` image and compare one row of the clean and the noisy image.

**What to expect.** The noisy row oscillates randomly around the clean one, with typical deviations of about ±20. The PSNR of the noisy image is about 22 dB.

**How to read the code.** `filters.add_gaussian_noise(image, sigma, rng=0)` adds noise; the fixed seed `rng=0` makes the result reproducible. `filters.psnr(clean, other)` returns the quality in decibels (higher = better).

In [ ]:
camera = images.load_test_image("camera")
noisy = filters.add_gaussian_noise(camera, 20.0, rng=0)
row = 200
plt.figure(figsize=(10, 3))
plt.plot(noisy[row], label="noisy", alpha=0.6)
plt.plot(camera[row], label="clean")
plt.xlabel("column [pixel]")
plt.ylabel("intensity")
plt.title(f"row {row}; PSNR of the noisy image {filters.psnr(camera, noisy):.1f} dB")
plt.legend()
plt.show()

## Gaussian and median filters

**What we do.** We filter the noisy image with the 3×3 and 5×5 Gaussian masks and with a 3×3 median filter, and show the PSNR of each result in the title.

**What to expect.** All three filters increase the PSNR. The 5×5 Gaussian removes more noise than the 3×3 one but makes the image visibly softer.

**How to read the code.** The dictionary `results` collects the images; the loop draws them in one row and computes `filters.psnr(camera, image)` for each title (`:.1f` prints one decimal).

In [ ]:
results = {
    "noisy": noisy,
    "Gauss 3x3": filters.gaussian_filter(noisy, 3),
    "Gauss 5x5": filters.gaussian_filter(noisy, 5),
    "median 3x3": filters.median_filter(noisy, 3),
}
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, image) in zip(axes, results.items(), strict=True):
    images.show_image(image, f"{title}: {filters.psnr(camera, image):.1f} dB", ax=ax)
plt.tight_layout()
plt.show()

## Gauss versus median at an edge

**What we do.** We use a synthetic image with sharp edges (a bright square of intensity 180 on a background of 80), add weak noise (σ = 5) and compare the filters along a row crossing the square.

**What to expect.** The Gaussian filter turns the jumps at columns 64 and 192 into ramps a few pixels wide (blurred edges); the median filter removes the noise on the flat parts and keeps the jumps sharp.

**How to read the code.** `filters.two_level_image()` returns the 256×256 test image; `noisy[row]` is the row through the middle of the square. We zoom into columns 40–90 with `plt.xlim` to see one edge in detail in the second panel.

In [ ]:
clean = filters.two_level_image()
noisy_square = filters.add_gaussian_noise(clean, 5.0, rng=1)
row = clean.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax in axes:
    ax.plot(noisy_square[row], label="noisy", alpha=0.5)
    ax.plot(filters.gaussian_filter(noisy_square, 5)[row], label="Gauss 5x5")
    ax.plot(filters.median_filter(noisy_square, 3)[row], label="median 3x3")
    ax.set_xlabel("column [pixel]")
    ax.set_ylabel("intensity")
    ax.legend()
axes[0].set_title("whole row")
axes[1].set_title("zoom on the left edge")
axes[1].set_xlim(40, 90)
plt.tight_layout()
plt.show()

## The gradient

**What we do.** We compute the two partial derivatives of the smoothed image with the Sobel masks and the length of the gradient.

**What to expect.** `gx` responds to vertical edges (changes from left to right), `gy` to horizontal edges (changes from top to bottom); both are positive or negative depending on the direction of the change, so we show them with the scaled display (grey = 0). The gradient magnitude is bright on all edges regardless of their direction.

**How to read the code.** We smooth first (`filters.gaussian_filter(camera, 5)`), because derivatives amplify noise. `edges.gradient` returns `(gx, gy)`, `edges.gradient_magnitude` returns `sqrt(gx² + gy²)`.

In [ ]:
smoothed = filters.gaussian_filter(camera, 5)
gx, gy = edges.gradient(smoothed)
magnitude = edges.gradient_magnitude(gx, gy)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
images.show_image(gx, "gx (left to right)", ax=axes[0], scaled=True)
images.show_image(gy, "gy (top to bottom)", ax=axes[1], scaled=True)
images.show_image(magnitude, "gradient magnitude", ax=axes[2], scaled=True)
plt.tight_layout()
plt.show()

## The Canny edge detector step by step

**What we do.** We thin the gradient magnitude to one-pixel-wide ridges (non-maximum suppression) and then select edge pixels with two thresholds (hysteresis): this is the complete Canny algorithm.

**What to expect.** The thinned magnitude contains thin lines along all edges, including weak ones caused by texture and noise; Canny keeps the strong edges and the weak pixels connected to them, giving clean contours.

**How to read the code.** `edges.quantize_direction(gx, gy)` rounds the gradient direction to 0°, 45°, 90° or 135°; `edges.non_maximum_suppression` uses it to compare each pixel with its two neighbours across the edge. `edges.canny(camera, 20, 50)` runs all steps (smoothing, gradient, thinning, hysteresis with thresholds 20 and 50) and returns a Boolean map, shown in black and white.

In [ ]:
thin = edges.non_maximum_suppression(magnitude, edges.quantize_direction(gx, gy))
edge_map = edges.canny(camera, 20, 50)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
images.show_image(thin, "after non-maximum suppression", ax=axes[0], scaled=True)
images.show_image(edge_map, "Canny edges (thresholds 20, 50)", ax=axes[1], scaled=True)
plt.tight_layout()
plt.show()

## Influence of the thresholds

**What we do.** We run Canny with three pairs of thresholds.

**What to expect.** Low thresholds give many edges (also grass texture), high thresholds only the strongest contours. Choosing them is a trade-off between missing edges and detecting noise.

**How to read the code.** The loop calls `edges.canny` with the pairs `(low, high)` and shows the number of edge pixels (`edge_map.sum()`, since `True` counts as 1) in the titles.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (low, high) in zip(axes, [(5, 15), (20, 50), (50, 120)], strict=True):
    edge_map = edges.canny(camera, low, high)
    images.show_image(
        edge_map, f"low {low}, high {high}: {edge_map.sum()} pixels", ax=ax, scaled=True
    )
plt.tight_layout()
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above, grouped by file, with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Three modules are used: `wavelets_lab.images` (loading and display), `wavelets_lab.filters` (noise, filters, PSNR) and `wavelets_lab.edges` (gradient and Canny). Functions marked *used internally* are called by the others.

**How to read the code.** Signatures follow Python syntax; "array-like" means anything `np.asarray` accepts.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `plt.plot(y, label, alpha)` | draws a curve; `alpha` is transparency | array → line |
| `ax.set_xlim(a, b)` | shows only `a ≤ x ≤ b` | numbers → – |
| `array.sum()` | sum of all entries (number of `True` for Boolean arrays) | array → number |
| `plt.subplots`, `ax.set_xlabel`, `ax.set_ylabel`, `ax.set_title`, `ax.legend` | layout and labels | – |

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.

### Module `wavelets_lab.filters` (file `src/wavelets_lab/filters.py`)

Noise, linear smoothing (convolution with a mask) and median filtering of images, plus quality measures. Images follow the convention of `wavelets_lab.images` (`float64` on `[0, 255]`). Imports:
```python
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
```
The module defines two constants from the lecture notes: `GAUSS_3X3 = [[1, 2, 1], [2, 4, 2], [1, 2, 1]] / 16` and `GAUSS_5X5 = [[2, 4, 5, 4, 2], [4, 9, 12, 9, 4], [5, 12, 15, 12, 5], [4, 9, 12, 9, 4], [2, 4, 5, 4, 2]] / 159`. Both are integer approximations of a Gaussian, divided by the sum of their entries so that they sum to 1 (a constant image stays unchanged).

#### `filters.add_gaussian_noise(image, sigma, rng=None)`

**What it is.** Adds white Gaussian noise: an independent normal random number with mean 0 and standard deviation `sigma` to every pixel. It models sensor and transmission noise.

**Input.**
- `image` (array-like): image.
- `sigma` (`float`): standard deviation of the noise, e.g. 20 (in intensity units).
- `rng` (`numpy.random.Generator`, `int` or `None`): random generator or seed; a fixed seed (e.g. `rng=0`) makes the example reproducible.

**Output.** `float64` noisy image of the same shape, **not** clipped (values may leave `[0, 255]`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. Create a generator: `generator = np.random.default_rng(rng)` (accepts a seed, an existing generator or `None`).
3. Draw standard normal numbers of the same shape: `generator.standard_normal(array.shape)`.
4. Return `array + sigma * noise`.

#### `filters.convolve2d(image, kernel)` (used internally)

**What it is.** 2D convolution of an image with a small mask: every output pixel is a weighted sum of the input pixels around it. At the border the image is extended by mirroring, so the output has the same shape as the input.

**Input.**
- `image` (array-like): greyscale image of shape `(R, C)`.
- `kernel` (array-like): 2D mask with **odd** side lengths `(k1, k2)`.

**Output.** `float64` array of shape `(R, C)`. Raises `ValueError` if the mask is not 2D with odd sides.

**How to write it yourself.**
1. Convert both arguments to `float64`; check `mask.ndim == 2` and that both sides are odd.
2. Pad the image by half the mask size on each side with mirroring: `np.pad(array, ((p0, p0), (p1, p1)), mode="symmetric")` with `p0, p1 = k1 // 2, k2 // 2`.
3. Take all windows of the mask's size: `windows = sliding_window_view(padded, mask.shape)` has shape `(R, C, k1, k2)`; `windows[i, j]` is the neighbourhood of pixel `(i, j)`.
4. Convolution uses the mask flipped in both directions, so compute `np.einsum("ijkl,kl->ij", windows, mask[::-1, ::-1])` (the sum over `k, l` of window times mask).
5. Return the result. (For symmetric masks such as the Gaussian ones, flipping changes nothing.)

#### `filters.gaussian_filter(image, size=3)`

**What it is.** Smooths (blurs) an image by convolution with the 3×3 or 5×5 Gaussian mask of the lecture notes. It weakens noise, but also softens edges.

**Input.**
- `image` (array-like): greyscale image.
- `size` (`3` or `5`, default 3): mask size; 5×5 smooths more.

**Output.** `float64` smoothed image of the same shape. Raises `ValueError` for other sizes.

**How to write it yourself.**
1. Put the two masks in a dictionary: `{3: GAUSS_3X3, 5: GAUSS_5X5}`.
2. If `size` is not a key, raise `ValueError("size must be 3 or 5")`.
3. Return `convolve2d(image, masks[size])`.

#### `filters.median_filter(image, size=3)`

**What it is.** Replaces every pixel by the median of the `size × size` neighbourhood around it. Unlike averaging, the median ignores a few outliers, so it removes impulse noise and keeps edges sharp.

**Input.**
- `image` (array-like): greyscale image.
- `size` (`int`, positive and odd, default 3).

**Output.** `float64` filtered image of the same shape. Raises `ValueError` for an even or non-positive size.

**How to write it yourself.**
1. Check that `size` is positive and odd.
2. Pad the image with `np.pad(array, size // 2, mode="symmetric")`.
3. Build all windows: `sliding_window_view(padded, (size, size))`, shape `(R, C, size, size)`.
4. Take the median over the last two axes: `np.median(windows, axis=(2, 3))`.

#### `filters.two_level_image(shape=(256, 256), low=80.0, high=180.0)`

**What it is.** A synthetic test image: a bright square (intensity `high`) covering the middle half of a dark background (intensity `low`). Its sharp edges make the difference between filters easy to see.

**Input.**
- `shape` (`tuple[int, int]`): image size.
- `low`, `high` (`float`): background and square intensities.

**Output.** `float64` image.

**How to write it yourself.**
1. `image = np.full(shape, low, dtype=np.float64)`.
2. With `rows, cols = shape`, set the central block: `image[rows // 4 : 3 * rows // 4, cols // 4 : 3 * cols // 4] = high`.
3. Return `image`.

#### `filters.mse(reference, image)` (used internally)

**What it is.** Mean squared error between two images of the same shape: `mean((reference - image) ** 2)`.

**Input.** Two array-like images of the same shape.

**Output.** `float`.

**How to write it yourself.**
1. Convert both to `float64` and subtract.
2. Square, take `np.mean`, and convert to a Python `float`.

#### `filters.psnr(reference, image, peak=255.0)`

**What it is.** Peak signal-to-noise ratio in decibels, `10 * log10(peak**2 / MSE)`, the standard measure of image quality: higher is better; +6 dB roughly means half the error amplitude. Identical images give infinity.

**Input.**
- `reference` (array-like): the clean image.
- `image` (array-like): the image to evaluate, same shape.
- `peak` (`float`, default 255): largest possible intensity.

**Output.** `float` (dB), `inf` if the images are equal.

**How to write it yourself.**
1. `error = mse(reference, image)`.
2. If `error == 0`, return `float("inf")`.
3. Return `10 * np.log10(peak**2 / error)` as a `float`.

### Module `wavelets_lab.edges` (file `src/wavelets_lab/edges.py`)

Image gradients and the Canny edge detector, following the steps of the lecture notes. Imports:
```python
import numpy as np
from scipy import ndimage
from wavelets_lab.filters import convolve2d, gaussian_filter
```
Constants: the Sobel masks `SOBEL_X = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]` (derivative along columns, i.e. in the x direction) and `SOBEL_Y = SOBEL_X.T` (derivative along rows, y direction, rows grow downwards). The notes call such masks Prewitt-type masks; Sobel masks additionally weight the middle row by 2.

#### `edges.gradient(image)`

**What it is.** Estimates the two partial derivatives of the image intensity with the Sobel masks.

**Input.**
- `image` (array-like): greyscale image.

**Output.** Tuple `(gx, gy)` of `float64` arrays with the image's shape: `gx` is the derivative from left to right, `gy` from top to bottom. For the ramp `image[i, j] = j` the Sobel estimate is `gx = 8` (the mask sums 1+2+1 differences of 2).

**How to write it yourself.**
1. Define `SOBEL_X` and `SOBEL_Y` as above.
2. We want correlation (mask applied as written), but `convolve2d` flips the mask; so pass a flipped mask to undo the flip: `gx = convolve2d(image, SOBEL_X[::-1, ::-1])`.
3. The same for `gy` with `SOBEL_Y`.
4. Return `gx, gy`.

#### `edges.gradient_magnitude(gx, gy)`

**What it is.** The length of the gradient vector, `sqrt(gx**2 + gy**2)`, i.e. how fast the intensity changes. It is large on edges.

**Input.** `gx`, `gy` (array-like): outputs of `gradient`.

**Output.** `float64` array.

**How to write it yourself.** Return `np.hypot(gx, gy)` (a numerically safe `sqrt(gx**2 + gy**2)`), converted to `float64`.

#### `edges.quantize_direction(gx, gy)`

**What it is.** Rounds the direction of the gradient to one of four directions: 0 = 0° (horizontal gradient, i.e. a vertical edge), 1 = 45°, 2 = 90°, 3 = 135°. Directions differing by 180° are treated as the same.

**Input.** `gx`, `gy` (array-like): outputs of `gradient`.

**Output.** Integer array with values 0, 1, 2, 3.

**How to write it yourself.**
1. Compute the angle in degrees with `np.degrees(np.arctan2(-gy, gx))`. The minus sign is needed because rows grow downwards, so "up" in the picture is negative `y`; with `-gy` angles are counted counter-clockwise as usual.
2. Reduce modulo 180 (`% 180.0`) to identify opposite directions.
3. Divide by 45, round with `np.rint`, convert to `int` and take `% 4` (so that 180° becomes 0 again).

#### `edges.non_maximum_suppression(magnitude, direction)`

**What it is.** Thins edges to a width of one pixel: a pixel is kept only if its gradient magnitude is at least as large as that of both neighbours along the gradient direction; all other pixels are set to 0.

**Input.**
- `magnitude` (array-like): gradient magnitude.
- `direction` (array-like): quantized directions from `quantize_direction`.

**Output.** `float64` array: the magnitude where it is a local maximum, 0 elsewhere.

**How to write it yourself.**
1. List the neighbour offsets `(d_row, d_col)` for the four directions: `((0, 1), (-1, 1), (-1, 0), (-1, -1))` for 0°, 45°, 90°, 135° (45° points up-right, so the row decreases).
2. Pad the magnitude with one row/column of zeros: `padded = np.pad(mag, 1, mode="constant")`.
3. For each direction index `k` with offset `(dr, dc)`: the neighbour in front of every pixel is the shifted slice `padded[1 + dr : 1 + dr + rows, 1 + dc : 1 + dc + cols]`, the neighbour behind is the slice with `-dr, -dc`.
4. Keep the pixels with `(direction == k) & (mag >= front) & (mag >= back)`; copy their magnitude into a zero array `result`.
5. Return `result`.

#### `edges.hysteresis_threshold(image, low, high)` (used internally)

**What it is.** Two-threshold selection of edge pixels. Pixels above `high` are certain edges; pixels above `low` are accepted only if they are connected (through 8-neighbours above `low`) to a certain edge. This keeps long edges continuous while rejecting isolated weak responses (noise).

**Input.**
- `image` (array-like): thinned gradient magnitude.
- `low`, `high` (`float`): thresholds with `low <= high`.

**Output.** Boolean array (edge map). Raises `ValueError` if `low > high`.

**How to write it yourself.**
1. Check `low <= high`.
2. `weak = values > low` marks candidates.
3. Label the connected components of `weak` using 8-connectivity: `labels, count = ndimage.label(weak, structure=np.ones((3, 3)))`; every component gets a number 1, 2, …, `count`.
4. If `count == 0`, return an all-`False` array.
5. Find the labels of components containing a strong pixel: `np.unique(labels[values > high])`, without the background label 0.
6. Return `np.isin(labels, strong_labels)`: whole components that touch a strong pixel.

(The lecture notes grow edges iteratively along the edge direction; labelling connected components gives the same idea in a few vectorized lines.)

#### `edges.canny(image, low=20.0, high=50.0, smoothing=5)`

**What it is.** The complete Canny edge detector.

**Input.**
- `image` (array-like): greyscale image on `[0, 255]`.
- `low`, `high` (`float`): hysteresis thresholds for the gradient magnitude.
- `smoothing` (`3` or `5`): size of the Gaussian mask used first.

**Output.** Boolean edge map of the image's shape.

**How to write it yourself.**
1. Smooth: `smoothed = gaussian_filter(image, size=smoothing)` (reduces noise, which would otherwise produce false edges).
2. Gradient: `gx, gy = gradient(smoothed)`.
3. Thin: `thin = non_maximum_suppression(gradient_magnitude(gx, gy), quantize_direction(gx, gy))`.
4. Select: return `hysteresis_threshold(thin, low, high)`.